
# Notebook 3 - Catálogo, busca e consultas

Este notebook replica a lógica central da aplicação web em ambiente analítico.


In [1]:

import pandas as pd
from pathlib import Path
BASE = Path("..").resolve()
df = pd.read_csv(BASE / "data/processed/base_integrada.csv")


## Etapa 1 - Definir funções de busca

In [2]:

def buscar_global(q="", entity="all"):
    q = str(q).strip()
    if not q:
        return df.head(20)
    mask = False
    if entity in ("all","cliente"):
        mask = mask | df["cliente_nome"].astype(str).str.contains(q, case=False, na=False)
    if entity in ("all","agencia"):
        mask = mask | df["nome_agencia"].astype(str).str.contains(q, case=False, na=False)
    if entity in ("all","produto"):
        mask = mask | df["produto_nome"].astype(str).str.contains(q, case=False, na=False)
    if entity in ("all","cpf"):
        mask = mask | df["cpf"].astype(str).str.contains(q, case=False, na=False)
    if entity in ("all","contrato") and q.isdigit():
        mask = mask | (df["id_contrato"] == int(q))
    return df[mask]


## Etapa 2 - Testar busca por cliente

In [3]:

exemplo = df["cliente_nome"].iloc[0].split()[0]
buscar_global(exemplo, "cliente").head()


,id_transacao,id_contrato,transacao_data,transacao_valor,transacao_tipo,id_cliente,id_produto,id_agencia,contrato_valor,contrato_data,cliente_nome,cpf,email,telefone,endereco,nome_agencia,cidade,estado,produto_nome,produto_tipo,taxa_juros,ano_mes,ano,mes,dia
0,1,14422,2024-09-16 23:50:58,10.16,Pagamento,812,27,65,7922.34,2026-01-04,Dr. João Miguel Pacheco,395.718.620-02,barrosluiz-miguel@example.net,+55 61 4651 3067,"Conjunto de Rodrigues, 74, Vila Petropolis, 14...",Agência 065,Sales,ES,Poupança 27,Poupança,2.02,2024-09,2024,9,16
1,2,6407,2024-04-12 18:00:52,12.20,Crédito,2254,7,60,93342.75,2025-04-26,André Melo,291.853.670-95,vpires@example.org,61 4518-1355,"Fazenda Farias, 583, Cinquentenário, 27080485 ...",Agência 060,Siqueira das Pedras,MG,Poupança 07,Poupança,2.64,2024-04,2024,4,12
17,18,8480,2024-09-30 05:35:03,21.01,Pix,860,40,48,72699.59,2025-04-11,Isabella Andrade,613.572.408-80,acaldeira@example.org,31 6417 3887,"Favela Gustavo Henrique Pires, 63, Indaiá, 195...",Agência 048,Carvalho do Galho,MG,Financiamento 40,Financiamento,4.31,2024-09,2024,9,30
20,21,8023,2024-12-29 19:05:47,7.94,Crédito,1933,23,46,25184.26,2024-09-16,Maria Clara Rodrigues,743.905.182-32,ana-livia37@example.net,+55 (071) 3563 1635,"Vila Nogueira, Vila Madre Gertrudes 3ª Seção, ...",Agência 046,Mendes da Praia,GO,Poupança 23,Poupança,2.64,2024-12,2024,12,29
25,26,1890,2025-09-12 00:18:15,10.10,Débito,2980,10,21,60820.82,2024-08-15,Dra. Isadora da Cunha,437.596.180-57,caroline18@example.net,(041) 7894-9989,"Distrito de da Rosa, 946, Outro, 39414641 Nogu...",Agência 021,Moura,SP,Poupança 10,Poupança,3.94,2025-09,2025,9,12


## Etapa 3 - Tabelas analíticas

In [4]:

produtos = (df[["id_contrato","produto_nome"]].drop_duplicates()
            .groupby("produto_nome")["id_contrato"].count()
            .sort_values(ascending=False).head(10))
clientes = df.groupby("cliente_nome")["id_transacao"].count().sort_values(ascending=False).head(10)
agencias = df.groupby("nome_agencia")["transacao_valor"].sum().sort_values(ascending=False).head(10)
produtos, clientes.head(), agencias.head()


(produto_nome
 Cartão 32            411
 Financiamento 29     395
 Empréstimo 06        390
 Conta Corrente 35    389
 Conta Corrente 15    388
 Empréstimo 05        386
 Conta Corrente 31    385
 Financiamento 37     385
 Financiamento 26     383
 Poupança 10          377
 Name: id_contrato, dtype: int64,
 cliente_nome
 Ana Caldeira            76
 Gael Henrique Fogaça    66
 Gabriel Câmara          66
 Bento Mendonça          65
 Clara Almeida           62
 Name: id_transacao, dtype: int64,
 nome_agencia
 Agência 026    13424.19
 Agência 058    13120.93
 Agência 073    13056.27
 Agência 027    12989.74
 Agência 018    12939.10
 Name: transacao_valor, dtype: float64)